# 面试题：追问或执行的 VOI 怎样设计？

可以直接回答：追问不是礼貌动作，而是信息价值决策。比较直接执行的期望效用，和付出追问成本后依据答案重新执行的期望效用；只有后者更高才打断用户。高风险且答案会改变动作的问题更值得追问，能从权威工具读取的信息不应追问。下面用采购助手的离线事件验证这一点。

## 真实案例

采购助手处理 6 条脱敏请求。`direct_success` 表示不追问时选对商品的概率，`resolved_success` 表示用户补充信息后选对的概率，`answer_rate` 表示用户会回复的概率。所有数值是教学实验的受控估计，不代表线上收益。

## 基线

先比较“始终执行”和按 VOI 追问。

## 结果解读

观察每条请求的直接效用、追问后效用和策略。

## 失败案例

再展示只按歧义数追问会忽略用户不回复与追问成本。

In [1]:
requests = [{'id':'P01','item':'显示器','direct_success':0.55,'resolved_success':0.96,'answer_rate':0.92,'gain':80,'loss':120,'ask_cost':4,'ambiguity':3}, {'id':'P02','item':'键盘','direct_success':0.88,'resolved_success':0.94,'answer_rate':0.75,'gain':20,'loss':30,'ask_cost':4,'ambiguity':2}, {'id':'P03','item':'云服务器','direct_success':0.42,'resolved_success':0.91,'answer_rate':0.90,'gain':200,'loss':300,'ask_cost':8,'ambiguity':4}, {'id':'P04','item':'鼠标','direct_success':0.90,'resolved_success':0.95,'answer_rate':0.35,'gain':15,'loss':20,'ask_cost':4,'ambiguity':3}, {'id':'P05','item':'VPN 续费','direct_success':0.70,'resolved_success':0.93,'answer_rate':0.95,'gain':70,'loss':100,'ask_cost':5,'ambiguity':2}, {'id':'P06','item':'会议室预订','direct_success':0.80,'resolved_success':0.98,'answer_rate':0.60,'gain':25,'loss':40,'ask_cost':3,'ambiguity':1}]  # 构造六条具有价格、风险和答复率的采购事件。
print('教学实验输入：id | 商品 | 歧义 | 直接成功率 | 用户答复率')  # 输出可读的原始事件字段。
for row in requests:  # 逐条预览代理面对的业务请求。
    print(row['id'], row['item'], row['ambiguity'], row['direct_success'], row['answer_rate'])  # 输出单条请求的决策特征。

教学实验输入：id | 商品 | 歧义 | 直接成功率 | 用户答复率
P01 显示器 3 0.55 0.92
P02 键盘 2 0.88 0.75
P03 云服务器 4 0.42 0.9
P04 鼠标 3 0.9 0.35
P05 VPN 续费 2 0.7 0.95
P06 会议室预订 1 0.8 0.6


In [2]:
baseline = {row['id']:'直接执行' for row in requests}  # 建立不考虑信息价值的始终执行基线。
baseline_utility = sum(row['direct_success'] * row['gain'] - (1 - row['direct_success']) * row['loss'] for row in requests)  # 计算基线在六条事件上的期望效用。
print('基线策略:', baseline)  # 输出始终执行的逐请求决策。
print('基线总期望效用:', round(baseline_utility, 2))  # 输出基线效用供后续公平比较。

基线策略: {'P01': '直接执行', 'P02': '直接执行', 'P03': '直接执行', 'P04': '直接执行', 'P05': '直接执行', 'P06': '直接执行'}
基线总期望效用: -43.5


In [3]:
def expected_utility(success, gain, loss):  # 定义成功收益和错误损失的期望效用函数。
    return success * gain - (1 - success) * loss  # 返回同一量纲下的执行期望价值。

def decide_by_voi(row):  # 根据追问信息价值选择动作。
    direct = expected_utility(row['direct_success'], row['gain'], row['loss'])  # 计算不追问时的直接执行效用。
    answered = expected_utility(row['resolved_success'], row['gain'], row['loss'])  # 计算拿到答案后执行的效用。
    asked = row['answer_rate'] * answered - row['ask_cost']  # 扣除答复概率和追问摩擦后计算追问效用。
    action = '追问' if asked > direct else '直接执行'  # 选择期望效用更高的动作。
    return {'id':row['id'], 'direct':direct, 'asked':asked, 'action':action}  # 返回可审计的中间量与决策。

In [4]:
decisions = [decide_by_voi(row) for row in requests]  # 对六条请求逐条执行手写 VOI 策略。
voi_utility = sum(max(item['direct'], item['asked']) for item in decisions)  # 汇总每条最优局部决策的期望效用。
print('id | 直接效用 | 追问效用 | VOI 决策')  # 输出包含中间量的结果表标题。
for item in decisions:  # 遍历每条可解释的策略结果。
    print(item['id'], round(item['direct'], 1), round(item['asked'], 1), item['action'])  # 输出两种选择与最终动作。
print('VOI 总期望效用:', round(voi_utility, 2), '提升:', round(voi_utility - baseline_utility, 2))  # 输出相对基线的教学实验提升。

id | 直接效用 | 追问效用 | VOI 决策
P01 -10.0 62.2 追问
P02 14.0 8.7 直接执行
P03 -90.0 131.5 追问
P04 11.5 0.6 直接执行
P05 19.0 50.2 追问
P06 12.0 11.2 直接执行
VOI 总期望效用: 281.44 提升: 324.94


In [5]:
naive_action = '追问' if requests[3]['ambiguity'] >= 3 else '直接执行'  # 构造只看歧义数的错误规则。
fixed_action = next(item['action'] for item in decisions if item['id'] == 'P04')  # 读取 P04 的 VOI 修正决策。
print('失败案例 P04：歧义数规则=', naive_action, '，VOI=', fixed_action)  # 展示低答复率使盲目追问不划算。
print('生产差距：线上需用历史完成率估计概率，并按金额、权限和用户群校准效用；未知时进入草稿或人工队列。')  # 说明离线小样本与生产控制面的差距。

失败案例 P04：歧义数规则= 追问 ，VOI= 直接执行
生产差距：线上需用历史完成率估计概率，并按金额、权限和用户群校准效用；未知时进入草稿或人工队列。


In [6]:
assert voi_utility > baseline_utility  # 验证 VOI 策略在本教学事件上优于始终执行。
assert fixed_action == '直接执行'  # 验证低答复率的 P04 不会被错误追问。
assert len(decisions) == 6  # 验证六条业务事件均产生了决策。